In [1]:
from pathlib import Path

import sqlite3
import pandas as pd

from agent import Agent

agent = Agent()

path = Path("data")

/Users/pedrojorge/Library/Caches/pypoetry/virtualenvs/desafio-artefact-WEcEnp-8-py3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
question = "Vocês vendem guitarra?"
response = agent.ask(question)

The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
No OCR engine found. Please review the install details.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
W1103 17:30:06.865000 24050 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 770/770 [00:00<00:00, 22289.65it/s]
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.


ModuleNotFoundError: No module named 'torchvision'

In [7]:
csv_files = list(path.glob("*.csv"))
dfs = [pd.read_csv(file) for file in csv_files]

In [8]:
dfs[5].head()

,category_id,name,description
0,1,Guitarras,"Guitarras elétricas de todos os estilos, do ro..."
1,2,Baixos,Baixos elétricos e acústicos para todas as for...
2,3,Baterias e Percussão,"Baterias acústicas, eletrônicas e peças avulsa..."
3,4,Teclados e Pianos,"Pianos digitais, sintetizadores, arranjadores ..."
4,5,Violões,"Violões clássicos, folk e eletroacústicos"


In [9]:
conn = sqlite3.connect("data/database.db")

for f in csv_files:
    df = pd.read_csv(f)

    table_name = f.stem.split(" - ", maxsplit=1)[1]
    
    df.to_sql(
        name=table_name,
        con=conn,
        if_exists="replace",
        index=False
    )

conn.close()

In [10]:
def query_db(query: str) -> pd.DataFrame:
    with sqlite3.connect("data/database.db") as conn:
        return pd.read_sql_query(query, conn)

In [11]:
query = """
SELECT
    p.product_id,
    p.name AS product_name,
    c.name AS category,
    SUM(oi.quantity) AS units_sold,
    SUM(oi.quantity * p.price_brl) AS estimated_revenue
FROM order_items oi
JOIN products p
    ON oi.product_id = p.product_id
JOIN categories c
    ON p.category_id = c.category_id
JOIN orders o
    ON oi.order_id = o.order_id
WHERE o.status != 'cancelled'
GROUP BY p.product_id, p.name, c.name
ORDER BY units_sold DESC;
"""

df = query_db(query)

In [12]:
df

,product_id,product_name,category,units_sold,estimated_revenue
0,130,Kala KA-ABP-CTG Barítono Cedar,Ukuleles,3,5247.0
1,106,Taylor 214ce Elétrico Aço Natural,Violões,2,14998.0
2,120,Kala KA-C Concert Mogno,Ukuleles,2,699.8
3,82,Yamaha C70 Nylon Natural,Violões,1,849.0
4,83,Tagima Memphis AC-39 Nylon Natural,Violões,1,429.9
5,93,Martin D-28 Dreadnought Natural,Violões,1,11499.0
6,94,Taylor 110e Dreadnought Natural,Violões,1,5999.0
7,96,Giannini GF-3D Dreadnought Sunburst,Violões,1,799.9
8,97,Crafter D-8 Dreadnought Natural,Violões,1,4599.0
9,119,Shelby SU-21S Soprano Sunburst,Ukuleles,1,159.9


In [13]:
# from dotenv import load_dotenv
# from langchain_groq import ChatGroq

# load_dotenv()

# llm = ChatGroq(
#     model="qwen/qwen3.8-27b", # Alternativa leve e mais rápida: "llama3-8b-8192"
#     temperature=0.2
# )

# resposta = llm.invoke("Você vende guitarras?")
# print(resposta.content)